Loading and Displaying Dataset

In [15]:
import pandas as pd

df = pd.read_csv("../data/sample_emails_with_triage_200.csv")
df.head()

,id,sender,subject,body,priority,triage_label,ideal_intent,ideal_tone
0,1,alerts@bank.com,Password Reset Request,Reminder: The client meeting is scheduled at 1...,low,notify_human,respond,neutral
1,2,alerts@bank.com,Congratulations! You've Won,Your invoice of INR 25515.09 is due on 2025-12...,low,respond,notify,neutral
2,3,no-reply@service.com,Promotion: Big Sale,Reminder: The client meeting is scheduled at 1...,low,ignore,respond,neutral
3,4,sales@shop.com,Monthly Report,"Hello team, please find the attached weekly re...",medium,respond,respond,neutral
4,5,no-reply@service.com,Survey,"Hello team, please find the attached weekly re...",low,respond,respond,neutral


Using Email Assistant Logic

In [16]:
def email_assistant(email_text):
    text = email_text.lower()

    if "urgent" in text or "schedule" in text or "deadline" in text or "submit" in text:
        return {"action": "notify", "tone": "urgent"}
    elif "congratulations" in text or "thank you" in text or "sale" in text:
        return {"action": "ignore", "tone": "polite"}
    else:
        return {"action": "respond", "tone": "neutral"}

Defining Dangerous Actions

In [17]:
DANGEROUS_ACTIONS = ["respond"]

HITL Checkpoint Logic

In [18]:
def hitl_check(action):
    if action in DANGEROUS_ACTIONS:
        return "WAIT_FOR_HUMAN"
    else:
        return "AUTO_APPROVED"

Simulating Human Approval

In [19]:
def human_desicion():
    desicion = input("Approve action? (yes/no): ")
    return desicion.lower() == "yes"

Applying HITL Check on the Data

In [20]:
results = []

for _, row in df.sample(200).iterrows():
    action, tone = email_assistant(row["body"])

    status = hitl_check(action)

    if status == "WAIT_FOR_HUMAN":
        approved = human_decision()
        final_action = action if approved else "blocked"
    else:
        final_action = action
    
    results.append({
        "email" : row["body"][:50],
        "ai_action" : action,
        "final_action": final_action,
        "hitl_status" : status
    })


pd.DataFrame(results)

,email,ai_action,final_action,hitl_status
0,Notice: Your account will be locked unless ver...,action,action,AUTO_APPROVED
1,Congratulations! You have been selected as a l...,action,action,AUTO_APPROVED
2,"Dear user, we detected a login from a new devi...",action,action,AUTO_APPROVED
3,Please complete the mandatory training module ...,action,action,AUTO_APPROVED
4,Security alert: multiple failed login attempts...,action,action,AUTO_APPROVED
...,...,...,...,...
195,Notice: Your account will be locked unless ver...,action,action,AUTO_APPROVED
196,"Hi, don't miss our sale with discounts up to 7...",action,action,AUTO_APPROVED
197,"Dear user, we detected a login from a new devi...",action,action,AUTO_APPROVED
198,Security alert: multiple failed login attempts...,action,action,AUTO_APPROVED


Saving the Output

In [21]:
results_df = pd.DataFrame(results)

results_df.to_csv(
    "../data/milestone3_output_swethanand-sangamreddi.csv", 
    index=False
)